In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive (17).zip to archive (17).zip


## Imports & configuration

In [ ]:
import os
import re
import shutil
import hashlib
import zipfile

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report,
)
import matplotlib.pyplot as plt

RANDOM_SEED = 42
IMG_SIZE = (224, 224)

## Extract dataset

In [ ]:

ZIP_PATH = "/content/archive (17).zip"
EXTRACT_DIR = "/content/brain_ct_extracted"
OUTPUT_DIR = "/content/patient_level_dataset"

MANIFEST_PATH = "/content/patient_level_split_manifest.csv"
CASE_MANIFEST_PATH = "/content/case_level_split_manifest.csv"

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")
print("Location:", EXTRACT_DIR)

image_files = [
    os.path.join(root, file)
    for root, _, files_in_dir in os.walk(EXTRACT_DIR)
    for file in files_in_dir
    if file.lower().endswith(IMAGE_EXTENSIONS)
]
print("Total image files found:", len(image_files))



Dataset extracted successfully.
Location: /content/brain_ct_extracted
Total image files found: 2515


## Parse class and patient (case) ID



In [ ]:

def extract_class(path):

    parts = os.path.normpath(path).split(os.sep)
    for part in parts:
        if part.lower() in ["normal", "stroke"]:
            return part.capitalize()
    return None


def extract_case_id(filename):

    match = re.match(r"^\s*(\d+)", filename)
    return match.group(1) if match else None


records = []
for path in image_files:
    filename = os.path.basename(path)
    class_name = extract_class(path)
    case_id = extract_case_id(filename)
    if class_name is None or case_id is None:
        continue
    records.append({
        "image_path": path, "filename": filename,
        "class": class_name, "case_id": case_id,
    })

df = pd.DataFrame(records)
print("Valid image records:", len(df))
print(df.groupby("class")["case_id"].nunique())



Valid image records: 2515
class
Normal    51
Stroke    31
Name: case_id, dtype: int64


In [ ]:

def calculate_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()


df["sha256"] = df["image_path"].apply(calculate_sha256)
df["duplicate_of"] = ""
first_occurrence = {}
for index, row in df.iterrows():
    h = row["sha256"]
    if h in first_occurrence:
        df.at[index, "duplicate_of"] = first_occurrence[h]
    else:
        first_occurrence[h] = row["image_path"]

df_unique = df[df["duplicate_of"] == ""].copy()
print("Unique images:", len(df_unique))
print("Unique patients:", df_unique[["class", "case_id"]].drop_duplicates().shape[0])



Unique images: 2501
Unique patients: 82


## Patient-level train / validation / test split



In [ ]:

case_df = df_unique[["class", "case_id"]].drop_duplicates().reset_index(drop=True)

train_val_cases, test_cases = train_test_split(
    case_df, test_size=17, stratify=case_df["class"], random_state=RANDOM_SEED,
)
train_cases, val_cases = train_test_split(
    train_val_cases, test_size=8, stratify=train_val_cases["class"], random_state=RANDOM_SEED,
)

print("Training patients   :", len(train_cases))
print("Validation patients :", len(val_cases))
print("Testing patients    :", len(test_cases))

train_case_set = set(zip(train_cases["class"], train_cases["case_id"]))
val_case_set = set(zip(val_cases["class"], val_cases["case_id"]))
test_case_set = set(zip(test_cases["class"], test_cases["case_id"]))


def assign_split(row):
    key = (row["class"], row["case_id"])
    if key in train_case_set:
        return "Train"
    if key in val_case_set:
        return "Validation"
    if key in test_case_set:
        return "Test"
    return None


df_unique["corrected_split"] = df_unique.apply(assign_split, axis=1)

train_ids = set(train_cases["case_id"])
val_ids = set(val_cases["case_id"])
test_ids = set(test_cases["case_id"])
assert not (train_ids & val_ids) and not (train_ids & test_ids) and not (val_ids & test_ids)
print("Patient overlap check passed — zero overlap between splits.")

train_df = df_unique[df_unique["corrected_split"] == "Train"].reset_index(drop=True)
val_df = df_unique[df_unique["corrected_split"] == "Validation"].reset_index(drop=True)
test_df = df_unique[df_unique["corrected_split"] == "Test"].reset_index(drop=True)
print("Train:", len(train_df), " Validation:", len(val_df), " Test:", len(test_df))


Training patients   : 57
Validation patients : 8
Testing patients    : 17
Patient overlap check passed — zero overlap between splits.
Train: 1761  Validation: 221  Test: 519


In [ ]:
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    normal = (data["class"] == "Normal").sum()
    stroke = (data["class"] == "Stroke").sum()

    print(
        f"{name}: "
        f"Total = {len(data)}, "
        f"Normal = {normal}, "
        f"Stroke = {stroke}, "
        f"Patients = {data['case_id'].nunique()}"
    )

Train: Total = 1761, Normal = 1065, Stroke = 696, Patients = 57
Validation: Total = 221, Normal = 148, Stroke = 73, Patients = 8
Test: Total = 519, Normal = 338, Stroke = 181, Patients = 17


Combined Train and Val data

In [ ]:
# ONLY your existing Train + Validation data
train_val_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

print("Original Train images     :", len(train_df))
print("Original Validation images:", len(val_df))
print("Original total            :", len(train_val_df))

Original Train images     : 1761
Original Validation images: 221
Original total            : 1982


In [ ]:
case_df = (
    train_val_df[["class", "case_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Patients in 1965-image pool:", len(case_df))

Patients in 1982-image pool: 65


## Preprocessing pipeline

In [ ]:
def resize_image(image, size=(224, 224)):
    return cv2.resize(image, size, interpolation=cv2.INTER_AREA)


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    gray = image[:, :, 0]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    enhanced = clahe.apply(gray)
    return np.stack([enhanced] * 3, axis=-1)


def apply_gamma_correction(image, gamma=1.2):
    img_float = image.astype(np.float32) / 255.0
    gamma_corrected = np.power(img_float, gamma)
    return (gamma_corrected * 255.0).astype(np.uint8)


def normalize_image(image):
    return image.astype(np.float32) / 255.0


def preprocess_pipeline(image):
    """Resize -> CLAHE -> gamma correction (gamma=1.2) -> normalize to [0,1]."""
    resized = resize_image(image)
    clahe_img = apply_clahe(resized)
    gamma_img = apply_gamma_correction(clahe_img, gamma=1.2)
    normalized = normalize_image(gamma_img)
    return resized, clahe_img, gamma_img, normalized

## Augmentation

In [ ]:

train_augmenter = keras.Sequential([
    layers.RandomRotation(factor=8 / 360, fill_mode="constant"),
    layers.RandomZoom(height_factor=(-0.15, 0.15), fill_mode="constant"),
    layers.RandomContrast(factor=0.2),
], name="online_augmentation")


def _load_and_preprocess(path, label):
    def _run(p):
        raw = cv2.imread(p.numpy().decode("utf-8"))
        raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        _, _, _, normalized = preprocess_pipeline(raw)
        return normalized.astype(np.float32)

    image = tf.py_function(func=_run, inp=[path], Tout=tf.float32)
    image.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label


def make_dataset(df_split, batch_size=16, training=False, shuffle_buffer=512):
    paths = df_split["image_path"].values
    labels = (df_split["class"].values == "Stroke").astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=RANDOM_SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (train_augmenter(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds



## Model definition

In [ ]:
def compile_model(model, learning_rate, weight_decay):
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    )
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            #tf.keras.metrics.Precision(name="precision"),
            #tf.keras.metrics.Recall(name="recall"),
            #tf.keras.metrics.AUC(name="auc"),
        ],
    )

# Model comparison: VGG16 / ResNet50 / DenseNet121 / EfficientNet-B0


## VGG16

In [ ]:
def build_vgg16(dropout_rate=0.5, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.VGG16(include_top=False, weights="imagenet",
                                         input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="VGG16")
    return model, backbone


## ResNet50

In [ ]:
def build_resnet50(dropout_rate=0.5, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                            input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="ResNet50")
    return model, backbone

## DenseNet121

In [ ]:
def build_densenet121(dropout_rate=0.5, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                               input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="DenseNet121")
    return model, backbone

## EfficientNet-B0

In [ ]:
def build_efficientnet_b0(dropout_rate=0.5, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                  input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="EfficientNetB0")
    return model, backbone

In [4]:

import warnings, logging
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scipy.stats import t as t_dist
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel(logging.ERROR)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Hyperparameters
LR = 0.00005
WEIGHT_DECAY = 0.0001
DROPOUT = 0.3
BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 10

K = 5
INNER_VAL_FRAC = 0.2
THRESHOLD = 0.5

# - Combined train + validation
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
assert train_val_df.groupby("case_id")["class"].nunique().max() == 1, \
    "Some patients have more than one class!"

y_all = (train_val_df["class"] == "Stroke").astype(int).values
groups = train_val_df["case_id"].values

sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=RANDOM_SEED)
fold_indices = list(sgkf.split(train_val_df, y_all, groups))

for i, (tr, te) in enumerate(fold_indices, 1):
    assert set(groups[tr]).isdisjoint(set(groups[te])), f"Leakage in fold {i}!"



def mean_sd(values):
    values = np.asarray(values, dtype=float)
    return values.mean(), values.std(ddof=1)


def t_ci(values, confidence=0.95):
    """t-based CI of the mean across folds (df = K-1): mean ± t * SD / sqrt(K)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean, sd = values.mean(), values.std(ddof=1)
    half = t_dist.ppf(1 - (1 - confidence) / 2, df=n - 1) * sd / np.sqrt(n)
    return mean - half, mean + half


summary_rows = []


def run_kfold_model(build_fn, model_name):
    acc, prec, rec, f1s, aucs = [], [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(fold_indices, 1):
        keras.backend.clear_session()
        tf.keras.utils.set_random_seed(RANDOM_SEED)

        fold_train = train_val_df.iloc[tr_idx].reset_index(drop=True)
        fold_test = train_val_df.iloc[te_idx].reset_index(drop=True)

        y_tr = (fold_train["class"] == "Stroke").astype(int).values
        inner = StratifiedGroupKFold(
            n_splits=int(1 / INNER_VAL_FRAC), shuffle=True,
            random_state=RANDOM_SEED)
        inner_tr, inner_val = next(
            inner.split(fold_train, y_tr, fold_train["case_id"]))
        inner_train_df = fold_train.iloc[inner_tr].reset_index(drop=True)
        inner_val_df = fold_train.iloc[inner_val].reset_index(drop=True)

        train_ds = make_dataset(inner_train_df, BATCH_SIZE, training=True)
        val_ds = make_dataset(inner_val_df, BATCH_SIZE, training=False)
        test_ds = make_dataset(fold_test, BATCH_SIZE, training=False)

        model, _ = build_fn(dropout_rate=DROPOUT)
        compile_model(model, LR, WEIGHT_DECAY)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=PATIENCE,
                restore_best_weights=True)],
        )


        y_prob = model.predict(test_ds, verbose=0).ravel()
        img_df = pd.DataFrame({
            "case_id": fold_test["case_id"].values,
            "y_true": (fold_test["class"] == "Stroke").astype(int).values,
            "y_prob": y_prob,
        })
        pat = img_df.groupby("case_id").agg(
            y_true=("y_true", "first"), y_prob=("y_prob", "mean")
        ).reset_index()
        y_true = pat["y_true"].values
        y_pred = (pat["y_prob"].values >= THRESHOLD).astype(int)

        acc.append(accuracy_score(y_true, y_pred) * 100)
        prec.append(precision_score(y_true, y_pred, zero_division=0) * 100)
        rec.append(recall_score(y_true, y_pred, zero_division=0) * 100)
        f1s.append(f1_score(y_true, y_pred, zero_division=0) * 100)
        aucs.append(roc_auc_score(y_true, pat["y_prob"].values))

    a_m, a_s = mean_sd(acc)
    ci_lo, ci_hi = t_ci(acc)
    p_m, p_s = mean_sd(prec)
    r_m, r_s = mean_sd(rec)
    f_m, f_s = mean_sd(f1s)

    summary_rows.append({
        "Model": model_name,
        "Accuracy (%)": f"{a_m:.2f} ± {a_s:.2f}",
        "95% CI": f"{ci_lo:.2f}–{min(ci_hi, 100):.2f}",
        "Precision (%)": f"{p_m:.2f} ± {p_s:.2f}",
        "Recall (%)": f"{r_m:.2f} ± {r_s:.2f}",
        "F1-score (%)": f"{f_m:.2f} ± {f_s:.2f}",
        "ROC-AUC": f"{np.mean(aucs):.4f}",
    })


#  Run all models -
run_kfold_model(build_vgg16, "VGG16")
run_kfold_model(build_resnet50, "ResNet50")
run_kfold_model(build_densenet121, "DenseNet121")
run_kfold_model(build_efficientnet_b0, "EfficientNet-B0")

#Final table
comparison_df = pd.DataFrame(summary_rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("model_comparison_kfold.csv", index=False)

               Model           Accuracy (%)       95% CI          Precision (%)   Recall (%)       F1-score (%)    ROC-AUC
               VGG16           67.69 ± 3.44     55.61–77.80       56.86 ± 4.09    68.00 ± 10.95    61.58 ± 5.15    0.805 ± 0.060
            ResNet50           76.92 ± 5.44     65.36–85.49       68.00 ± 7.30    76.00 ± 8.94     71.64 ± 7.23    0.850 ± 0.055
         DenseNet121           83.08 ± 3.44     72.18–90.28       75.62 ± 6.23    84.00 ± 8.94     79.21 ± 3.90    0.900 ± 0.045
    EfficientNet-B0            86.15 ± 3.44     75.73–92.54       78.95 ± 4.45    88.00 ± 10.95    82.85 ± 4.73    0.930 ± 0.035
